# Creating a Migration Algorithm

This tutorial demonstrates how we can create a simple migration algorithm on EdgeSimPy.

First, let's import the EdgeSimPy modules:

In [9]:
try:
    # Importing EdgeSimPy components
    from edge_sim_py import *
    import networkx as nx
    import msgpack

    # Importing Matplotlib, Pandas, and NumPy for logs parsing and visualization
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np

except ModuleNotFoundError:
    # Downloading EdgeSimPy binaries from GitHub (the "-q" parameter suppresses Pip's output. You check the full logs by removing it)
    %pip install -q git+https://github.com/EdgeSimPy/EdgeSimPy.git

    # Downloading Pandas, NumPy, and Matplotlib (these are not directly used here, but they can be useful for logs parsing and visualization)
    %pip install -q pandas
    %pip install -q numpy
    %pip install -q matplotlib

    # Importing EdgeSimPy components and its built-in libraries (NetworkX and MessagePack)
    from edge_sim_py import *
    import networkx as nx
    import msgpack

    # Importing Matplotlib, Pandas, and NumPy for logs parsing and visualization
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np

## Implementing the Migration Algorithm

Our simple migration algorithm will work according to the well-known Worst-Fit heuristic. In a nutshell, it will provision each service to the edge server with the largest amount of free resources.

In [10]:
def my_algorithm(parameters):
    # Let's iterate over the list of services using the 'all()' helper method
    print("\n\n")
    print(f"==== TIME STEP {parameters['current_step']} ====")
    print("---- EDGE SERVERS ----")
    for server in EdgeServer.all():
        server_metadata = {
            "server": server,
            "capacity": [server.cpu, server.memory, server.disk],
            "demand": [server.cpu_demand, server.memory_demand, server.disk_demand],
            "container_layers": [layer.id for layer in server.container_layers],
            "services": [service for service in server.services],
        }
        print(server_metadata)

    print("")
    print("---- SERVICES ----")
    for service in Service.all():
        service_image = ContainerImage.find_by(attribute_name="digest", attribute_value=service.image_digest)
        service_layers = [ContainerLayer.find_by(attribute_name="digest", attribute_value=layer).id for layer in service_image.layers_digests]

        service_metadata = {
            "service": service,
            "requirements": [service.cpu_demand, service.memory_demand],
            "layers": service_layers,
            "server": service.server,
            "available": service._available,
            "being_provisioned": service.being_provisioned,
        }
        print(service_metadata)

    # We don't want to migrate services are are already being migrated. As we want to avoid excessive migrations, we are going to
    # define a cool down period for services that have been migrated. For simplicity, we are going to set the cool down period as
    # 10 time steps. To get such information, we are going to access the service's last migration time step and compare it with
    # the current time step. If the difference between the current time step and the last migration time step is less than 10,
    # we are going to skip the service.
    COOLDOWN_STEPS = 10

    for service in Service.all():
        # Gathering the current time step and the last migration time step of the service
        current_step = parameters["current_step"]

        if len(service._Service__migrations) > 0 and service._Service__migrations[-1]["end"] is not None:
            last_migration_ended_at = service._Service__migrations[-1]["end"]
        else:
            last_migration_ended_at = None

        # Checking if the service is not being provisioned and if the cool down period has being reached
        service_has_not_been_migrated_yet = len(service._Service__migrations) == 0
        service_not_being_provisioned = service.being_provisioned is False
        cooldown_has_been_reached = last_migration_ended_at is not None and current_step - last_migration_ended_at >= COOLDOWN_STEPS

        if service_has_not_been_migrated_yet or service_not_being_provisioned and cooldown_has_been_reached:
            # We need to sort edge servers based on amount of free resources they have. To do so, we are going to use Python's
            # "sorted" method (you can learn more about "sorted()" in this link: https://docs.python.org/3/howto/sorting.html).
            # The capacity of edge servers is modeled in three layers (CPU, memory, and disk). For simplicity, we are going to
            # use only the CPU layer to sort edge servers. We calculate the free CPU resources of each edge server by subtracting
            # the CPU demand of the services hosted in the edge server from the total CPU capacity of the edge server. We then
            # sort edge servers based on their free CPU resources. To do so, we use the "sorted" method and set the "key" attribute
            # as a lambda function that calculates the free CPU resources of edge servers. We set the "reverse" attribute as "True"
            # as we want to sort edge servers by their free CPU resources in descending order.
            edge_servers = sorted(
                EdgeServer.all(),
                key=lambda s: s.cpu - s.cpu_demand,
                reverse=True,
            )

            for edge_server in edge_servers:
                # Checking if the edge server has resources to host the service
                if edge_server.has_capacity_to_host(service=service):
                    # We just need to migrate the service if it's not already in the least occupied edge server
                    if service.server != edge_server:
                        print(f"\t\t[STEP {parameters['current_step']}] Migrating {service} From {service.server} to {edge_server}")

                        service.provision(target_server=edge_server)

                        # After start migrating the service we can move on to the next service
                        break

## Running the Simulation

Before testing our algorithm, we must tell EdgeSimPy when it must stop the simulation. For example, let's run the simulation for 600 seconds (i.e., 10 minutes).

To do so, we must create a simple function that will be used as the simulation's stopping criterion. Behind the scenes, at the end of each time step, EdgeSimPy will run that function, halting the simulation if it returns `True`.

In [11]:
def stopping_criterion(model: object):
    # As EdgeSimPy will halt the simulation whenever this function returns True,
    # its output will be a boolean expression that checks if the current time step is 600
    return model.schedule.steps == 600

Once we have developed our stopping criterion function, we can create an instance of the `Simulation` class passing a couple of arguments, load a dataset, and run the simulation to check how our algorithm goes.

In [12]:
# Creating a Simulator object
simulator = Simulator(
    tick_duration=1,
    tick_unit="seconds",
    stopping_criterion=stopping_criterion,
    resource_management_algorithm=my_algorithm,
)

# Loading a sample dataset from GitHub
simulator.initialize(input_file="https://raw.githubusercontent.com/EdgeSimPy/edgesimpy-tutorials/master/datasets/sample_dataset1.json")

# Executing the simulation
simulator.run_model()

串流輸出內容已截斷至最後 5000 行。
{'service': Service_4, 'requirements': [1, 2048], 'layers': [3, 4], 'server': EdgeServer_6, 'available': True, 'being_provisioned': False}
{'service': Service_5, 'requirements': [1, 2048], 'layers': [5], 'server': EdgeServer_5, 'available': True, 'being_provisioned': False}
{'service': Service_6, 'requirements': [1, 2048], 'layers': [5], 'server': EdgeServer_6, 'available': True, 'being_provisioned': False}



==== TIME STEP 346 ====
---- EDGE SERVERS ----
{'server': EdgeServer_1, 'capacity': [8, 16384, 131072], 'demand': [1, 2048, 82], 'container_layers': [39, 38, 42, 44], 'services': [Service_2]}
{'server': EdgeServer_2, 'capacity': [8, 16384, 131072], 'demand': [0, 0, 82], 'container_layers': [40, 43, 46, 45], 'services': []}
{'server': EdgeServer_3, 'capacity': [8, 8192, 131072], 'demand': [0, 0, 27], 'container_layers': [41], 'services': []}
{'server': EdgeServer_4, 'capacity': [8, 8192, 131072], 'demand': [0, 0, 80], 'container_layers': [33, 34, 35], 'service

In [13]:
def my_load_balancing_migration(parameters):
    """
    主動負載均衡 Migration 策略：
    - 計算每個 server 的 CPU 利用率
    - 優先把 service 從高負載 server 遷移到低負載（空閒）server
    - 加入 cooldown 避免過度遷移
    """
    current_step = parameters["current_step"]

    COOLDOWN = 15  # 遷移後冷卻時間

    # 初始化 cooldown
    if not hasattr(my_load_balancing_migration, "cooldown"):
        my_load_balancing_migration.cooldown = {}

    print(f"\n==== TIME STEP {current_step} ====")

    # 計算利用率
    server_utils = {}
    for server in EdgeServer.all():
        util = server.cpu_demand / server.cpu if server.cpu > 0 else 0
        server_utils[server] = util
        print(f"{server}: CPU util = {util:.2f}, Services = {[str(s) for s in server.services]}")

    # 按利用率由高到低排序
    sorted_servers = sorted(server_utils.keys(), key=lambda s: server_utils[s], reverse=True)

    # 最空閒的 server
    least_loaded = min(server_utils, key=server_utils.get)

    # 如果最空閒 server 利用率已經接近平均，就唔使再遷
    avg_util = sum(server_utils.values()) / len(server_utils)
    if server_utils[least_loaded] >= avg_util - 0.05:
        print("負載已經較為均衡，本步不做遷移")
        return

    # 嘗試從高負載 server 遷移 service 到最空閒 server
    migrated = False
    for server in sorted_servers:
        if server == least_loaded:
            continue
        if server_utils[server] <= server_utils[least_loaded]:
            continue

        services_to_try = list(server.services)
        for service in services_to_try:
            # cooldown 檢查
            last_migrate = my_load_balancing_migration.cooldown.get(service, -999)
            if current_step - last_migrate < COOLDOWN:
                continue

            if least_loaded.has_capacity_to_host(service=service):
                print(f"[STEP {current_step}] 遷移 {service} 從 {service.server} → {least_loaded}")
                service.provision(target_server=least_loaded)
                my_load_balancing_migration.cooldown[service] = current_step
                migrated = True
                break  # 每次只遷一個

        if migrated:
            break

    if not migrated:
        print("本步沒有合適的遷移")

In [14]:
def stopping_criterion(model):
    return model.schedule.steps == 200

simulator = Simulator(
    tick_duration=1,
    tick_unit="seconds",
    stopping_criterion=stopping_criterion,
    resource_management_algorithm=my_load_balancing_migration,
)

simulator.initialize(input_file="https://raw.githubusercontent.com/EdgeSimPy/edgesimpy-tutorials/master/datasets/sample_dataset1.json")

print("開始運行主動負載均衡 Migration 策略...")
simulator.run_model()
print("模擬結束！")

開始運行主動負載均衡 Migration 策略...

==== TIME STEP 1 ====
EdgeServer_1: CPU util = 0.00, Services = []
EdgeServer_2: CPU util = 0.00, Services = []
EdgeServer_3: CPU util = 0.00, Services = []
EdgeServer_4: CPU util = 0.25, Services = ['Service_4', 'Service_5']
EdgeServer_5: CPU util = 0.25, Services = ['Service_1', 'Service_2']
EdgeServer_6: CPU util = 0.17, Services = ['Service_3', 'Service_6']
[STEP 1] 遷移 Service_4 從 EdgeServer_4 → EdgeServer_1

==== TIME STEP 2 ====
EdgeServer_1: CPU util = 0.12, Services = []
EdgeServer_2: CPU util = 0.00, Services = []
EdgeServer_3: CPU util = 0.00, Services = []
EdgeServer_4: CPU util = 0.25, Services = ['Service_4', 'Service_5']
EdgeServer_5: CPU util = 0.25, Services = ['Service_1', 'Service_2']
EdgeServer_6: CPU util = 0.17, Services = ['Service_3', 'Service_6']
[STEP 2] 遷移 Service_5 從 EdgeServer_4 → EdgeServer_2

==== TIME STEP 3 ====
EdgeServer_1: CPU util = 0.12, Services = []
EdgeServer_2: CPU util = 0.12, Services = []
EdgeServer_3: CPU util = 0

In [15]:
print("=== 最終 Edge Servers 狀態 ===")
for s in EdgeServer.all():
    util = s.cpu_demand / s.cpu if s.cpu > 0 else 0
    print(f"{s} | CPU: {s.cpu_demand}/{s.cpu} ({util:.2%}) | Mem: {s.memory_demand}/{s.memory} | Services: {[str(svc) for svc in s.services]}")

=== 最終 Edge Servers 狀態 ===
EdgeServer_1 | CPU: 1/8 (12.50%) | Mem: 2048/16384 | Services: ['Service_4']
EdgeServer_2 | CPU: 1/8 (12.50%) | Mem: 2048/16384 | Services: ['Service_5']
EdgeServer_3 | CPU: 1/8 (12.50%) | Mem: 2048/8192 | Services: ['Service_1']
EdgeServer_4 | CPU: 1/8 (12.50%) | Mem: 2048/8192 | Services: ['Service_2']
EdgeServer_5 | CPU: 1/12 (8.33%) | Mem: 1024/16384 | Services: []
EdgeServer_6 | CPU: 2/12 (16.67%) | Mem: 4096/16384 | Services: ['Service_3', 'Service_6']


In [16]:
# ========== 官方 Worst-Fit 對比實驗 ==========
def worst_fit_migration(parameters):
    current_step = parameters["current_step"]
    print(f"\n==== TIME STEP {current_step} (Worst-Fit) ====")

    for service in Service.all():
        if service.server is None or service.being_provisioned:
            continue

        # 按剩餘 CPU 由多到少排序（Worst-Fit）
        edge_servers = sorted(
            EdgeServer.all(),
            key=lambda s: s.cpu - s.cpu_demand,
            reverse=True
        )

        for edge_server in edge_servers:
            if edge_server.has_capacity_to_host(service=service):
                if service.server != edge_server:
                    print(f"[STEP {current_step}] Migrating {service} From {service.server} to {edge_server}")
                    service.provision(target_server=edge_server)
                break

def stopping_criterion(model):
    return model.schedule.steps == 200

simulator = Simulator(
    tick_duration=1,
    tick_unit="seconds",
    stopping_criterion=stopping_criterion,
    resource_management_algorithm=worst_fit_migration,
)

simulator.initialize(input_file="https://raw.githubusercontent.com/EdgeSimPy/edgesimpy-tutorials/master/datasets/sample_dataset1.json")

print("開始運行官方 Worst-Fit Migration...")
simulator.run_model()
print("Worst-Fit 模擬結束！")

print("\n=== Worst-Fit 最終 Edge Servers 狀態 ===")
for s in EdgeServer.all():
    util = s.cpu_demand / s.cpu if s.cpu > 0 else 0
    print(f"{s} | CPU: {s.cpu_demand}/{s.cpu} ({util:.2%}) | Mem: {s.memory_demand}/{s.memory} | Services: {[str(svc) for svc in s.services]}")

開始運行官方 Worst-Fit Migration...

==== TIME STEP 1 (Worst-Fit) ====
[STEP 1] Migrating Service_1 From EdgeServer_5 to EdgeServer_6
[STEP 1] Migrating Service_3 From EdgeServer_6 to EdgeServer_5
[STEP 1] Migrating Service_4 From EdgeServer_4 to EdgeServer_6
[STEP 1] Migrating Service_5 From EdgeServer_4 to EdgeServer_1
[STEP 1] Migrating Service_6 From EdgeServer_6 to EdgeServer_2

==== TIME STEP 2 (Worst-Fit) ====
[STEP 2] Migrating Service_2 From EdgeServer_5 to EdgeServer_6
[STEP 2] Migrating Service_3 From EdgeServer_5 to EdgeServer_3
[STEP 2] Migrating Service_4 From EdgeServer_6 to EdgeServer_5

==== TIME STEP 3 (Worst-Fit) ====
[STEP 3] Migrating Service_1 From EdgeServer_6 to EdgeServer_5
[STEP 3] Migrating Service_4 From EdgeServer_5 to EdgeServer_6

==== TIME STEP 4 (Worst-Fit) ====
[STEP 4] Migrating Service_2 From EdgeServer_6 to EdgeServer_5

==== TIME STEP 5 (Worst-Fit) ====
[STEP 5] Migrating Service_1 From EdgeServer_5 to EdgeServer_6
[STEP 5] Migrating Service_2 From EdgeS